# Get around - analyse

Ce notebook analyse le dataset des retards de restitution (`get_around_delay_analysis.xlsx`) pour répondre aux questions du Product Manager sur la mise en place d'un **délai minimum entre deux locations** : quel seuil retenir, et sur quel périmètre de véhicules (tout le parc ou seulement les voitures Connect).

In [9]:
import pandas as pd
import plotly.express as px


## Delay analysis

On étudie ici les retards à la restitution des véhicules (`delay_at_checkout_in_minutes`) et leur impact sur la location suivante, afin de simuler l'effet d'un délai minimum imposé entre deux réservations.

### Chargement et exploration initiale

#### Structure du dataset, valeurs manquantes

In [10]:
dfda = pd.read_excel("data/get_around_delay_analysis.xlsx")
print(dfda.shape)
dfda.head()

(21310, 7)


,rental_id,car_id,checkin_type,state,delay_at_checkout_in_minutes,previous_ended_rental_id,time_delta_with_previous_rental_in_minutes
0,505000,363965,mobile,canceled,NaN,NaN,NaN
1,507750,269550,mobile,ended,-81.0,NaN,NaN
2,508131,359049,connect,ended,70.0,NaN,NaN
3,508865,299063,connect,canceled,NaN,NaN,NaN
4,511440,313932,mobile,ended,NaN,NaN,NaN


In [3]:
dfda.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21310 entries, 0 to 21309
Data columns (total 7 columns):
 #   Column                                      Non-Null Count  Dtype  
---  ------                                      --------------  -----  
 0   rental_id                                   21310 non-null  int64  
 1   car_id                                      21310 non-null  int64  
 2   checkin_type                                21310 non-null  object 
 3   state                                       21310 non-null  object 
 4   delay_at_checkout_in_minutes                16346 non-null  float64
 5   previous_ended_rental_id                    1841 non-null   float64
 6   time_delta_with_previous_rental_in_minutes  1841 non-null   float64
dtypes: float64(3), int64(2), object(2)
memory usage: 1.1+ MB


In [4]:
dfda.isnull().sum()

rental_id                                         0
car_id                                            0
checkin_type                                      0
state                                             0
delay_at_checkout_in_minutes                   4964
previous_ended_rental_id                      19469
time_delta_with_previous_rental_in_minutes    19469
dtype: int64

---
Sur les 21 310 locations enregistrées :
 - 4 964 (23.3%) n'ont pas de date de restitution renseignée 
 - 1 841 (8.6%) sont chaînées à une location précédente du même véhicule à moins de 12h d'écart

In [11]:
dfda[dfda['delay_at_checkout_in_minutes'].isna()]['checkin_type'].value_counts()

checkin_type
mobile     4059
connect     905
Name: count, dtype: int64

#### répartition des locations par état (`state`)

In [12]:
state_counts = dfda['state'].value_counts().rename_axis('state').reset_index(name='count')
print(state_counts)
fig = px.pie(state_counts, names='state', values='count', title='Répartition des états de location', width=400)
fig.update_layout(legend=dict(orientation='h', yanchor='top', y=0.05, xanchor='center', x=0.5))
fig.show()

      state  count
0     ended  18045
1  canceled   3265


#### répartition des locations par mode de checkin (`checkin_type`)

In [13]:
checkin_counts = dfda['checkin_type'].value_counts().rename_axis('checkin_type').reset_index(name='count')
print(checkin_counts)
fig = px.pie(checkin_counts, names='checkin_type', values='count', title='Répartition des types de checkin', width=400)
fig.update_layout(legend=dict(orientation='h', yanchor='top', y=0.05, xanchor='center', x=0.5))
fig.show()

  checkin_type  count
0       mobile  17003
1      connect   4307


On croise l'absence de donnée sur la restitution `delay_at_checkout_in_minutes`null, avec l'état de la location `state`

In [14]:
pd.crosstab(dfda['state'], dfda['delay_at_checkout_in_minutes'].isnull(), rownames=['state'], colnames=['delay_is_null'])

delay_is_null,False,True
state,,
canceled,1,3264
ended,16345,1700


L'absence de délai de restitution est quasi intégralement expliqué par l'annulation de la location

### Nettoyage : `is_late`

On dérive `is_late` (booléen nullable) sans imputer les ~4 965 lignes sans delay renseigné (annulations + quelques `ended` non trackées), pour ne pas biaiser le taux de retard réel. On garde aussi les valeurs brutes de delay pour tous les calculs

In [15]:
# is_late reste NA quand le delay n'est pas renseigné (pas d'imputation à "à l'heure")
dfda['is_late'] = (dfda['delay_at_checkout_in_minutes'] > 0).astype('boolean')
dfda.loc[dfda['delay_at_checkout_in_minutes'].isna(), 'is_late'] = pd.NA

dfda['is_late'].value_counts(dropna=False)

is_late
True     9404
False    6942
<NA>     4964
Name: count, dtype: Int64

In [16]:
is_late_counts = dfda['is_late'].value_counts(dropna=False).rename_axis('is_late').reset_index(name='count')
is_late_counts['is_late'] = is_late_counts['is_late'].astype('object').map({True: 'en retard', False: 'à l\'heure'}).fillna('inconnu')
print(is_late_counts)

# rouge et vert = 2e et 3e couleurs de la palette qualitative Plotly par défaut (px.colors.qualitative.Plotly)
is_late_colors = {'en retard': '#EF553B', 'à l\'heure': '#00CC96', 'inconnu': '#BAB0AC'}
fig = px.pie(
    is_late_counts, names='is_late', values='count', color='is_late',
    color_discrete_map=is_late_colors, title='Répartition des locations par retard (is_late)', width=400
)
fig.update_layout(legend=dict(orientation='h', yanchor='top', y=0.05, xanchor='center', x=0.5))
fig.show()

     is_late  count
0  en retard   9404
1  à l'heure   6942
2    inconnu   4964


In [17]:
is_late_counts = dfda.loc[~dfda['is_late'].isna(),'is_late'].value_counts(dropna=False).rename_axis('is_late').reset_index(name='count')
is_late_counts['is_late'] = is_late_counts['is_late'].astype('object').map({True: 'en retard', False: 'à l\'heure'}).fillna('inconnu')
print(is_late_counts)
is_late_colors = {'en retard': '#EF553B', 'à l\'heure': '#00CC96', 'inconnu': '#BAB0AC'}
fig = px.pie(
    is_late_counts, names='is_late', values='count', color='is_late',
    color_discrete_map=is_late_colors, title='Répartition des locations par retard (is_late)', width=400
)
fig.update_layout(legend=dict(orientation='h', yanchor='top', y=0.05, xanchor='center', x=0.5))
fig.show()

     is_late  count
0  en retard   9404
1  à l'heure   6942


## Distributions

analye et forme de la distribution des délais, en comparant mobile vs connect.

In [18]:
# mapping explicite pour garder la même couleur par checkin_type sur tous les graphiques,
# indépendamment de l'ordre des catégories (groupby trie alphabétiquement, pas les autres)
checkin_colors = {'mobile': px.colors.qualitative.Plotly[0], 'connect': px.colors.qualitative.Plotly[1]}

In [19]:
print(dfda['delay_at_checkout_in_minutes'].describe())
print()
print(dfda['delay_at_checkout_in_minutes'].quantile([0.01, 0.05, 0.5, 0.95, 0.99]))

count    16346.000000
mean        59.701517
std       1002.561635
min     -22433.000000
25%        -36.000000
50%          9.000000
75%         67.000000
max      71084.000000
Name: delay_at_checkout_in_minutes, dtype: float64

0.01    -853.00
0.05    -230.00
0.50       9.00
0.95     397.75
0.99    1490.55
Name: delay_at_checkout_in_minutes, dtype: float64


pour afficher la distribution on limite le delai de restitution à [-500,500] excluant des outliers de remise prématuré et englobant plus 95% des retards

In [20]:
mask_delay_clipped = dfda['delay_at_checkout_in_minutes'].between(-500, 500)
px.histogram(
    dfda[mask_delay_clipped], x='delay_at_checkout_in_minutes', color='checkin_type',
    color_discrete_map=checkin_colors, marginal='box', nbins=100,
    title='Distribution du delay au checkout (limité à [-500, 500] minutes)'
).show()

In [54]:
# histnorm='probability density' : chaque distribution est ramenée à une aire de 1,
# ce qui rend les formes comparables malgré des effectifs très différents (17003 mobile vs 4307 connect)
fig = px.histogram(
    dfda[mask_delay_clipped], x='delay_at_checkout_in_minutes', color='checkin_type',
    color_discrete_map=checkin_colors, histnorm='probability density', barmode='overlay', opacity=0.6,
    nbins=100, title='Distribution normalisée du delay au checkout par type de check-in'
)
fig.update_xaxes(title='delay au checkout (min)')
fig.update_yaxes(title='densité de probabilité')
fig.show()

**distribution des retards par checkin**

In [51]:
print(dfda[dfda['is_late'] == True].groupby('checkin_type')['delay_at_checkout_in_minutes'].describe())
print()
print(dfda[dfda['is_late'] == True].groupby('checkin_type')['delay_at_checkout_in_minutes'].quantile([0.01, 0.05, 0.5, 0.95, 0.99]))

               count        mean          std  min   25%   50%    75%      max
checkin_type                                                                  
connect       1459.0   80.109664   134.755238  1.0  16.0  41.0   94.0   1466.0
mobile        7945.0  224.136816  1350.615764  1.0  20.0  56.0  142.0  71084.0

checkin_type      
connect       0.01       1.00
              0.05       3.00
              0.50      41.00
              0.95     253.10
              0.99     644.78
mobile        0.01       1.00
              0.05       4.00
              0.50      56.00
              0.95     841.60
              0.99    2729.48
Name: delay_at_checkout_in_minutes, dtype: float64


In [21]:
late_by_checkin = dfda.dropna(subset=['is_late']).groupby('checkin_type')['is_late'].mean().reset_index()
late_by_checkin['pct_label'] = late_by_checkin['is_late'].apply(lambda x: f"{x:.1%}")
fig = px.bar(
    late_by_checkin, x='checkin_type', y='is_late', color='checkin_type',
    color_discrete_map=checkin_colors, text='pct_label',
    title='Part des locations en retard par type de checkin', width=450
)
fig.update_traces(textposition='outside')
fig.show()

---
**Conclusion** : les délais se concentrent autour de 0 minute (médiane 9 min, tous statuts confondus), mais avec une queue de distribution extrême côté mobile (jusqu'à 71 084 min, ~49 jours), d'où l'affichage borné à [-500,500]. En se limitant aux retards réels (`is_late=True`, <500 min, soit >95% de l'échantillon), c'est en fait **mobile** qui a la distribution la plus étalée et les retards typiquement les plus longs (médiane 48 min, moyenne 82.8, écart-type 95.0) contre connect (médiane 41 min, moyenne 68.6, écart-type 78.8) — l'inverse de la première impression visuelle. Connect reste néanmoins moins fréquemment en retard (43% vs 61% en mobile) : des retards plus rares, mais pas plus courts une fois qu'ils surviennent.

## Locations chaînées : impact sur le conducteur suivant

Seules les locations avec `previous_ended_rental_id` renseigné (<12h après la précédente location du même véhicule) peuvent être concrètement retardées au checkin par un retard de restitution. Démarche en 4 temps :
1. isoler ce sous-ensemble (`chained`)
2. récupérer, par auto-jointure, le delay réel de la location précédente
3. en déduire si le checkin suivant a été réellement impacté (`checkin_impacted`)
4. comparer le taux d'annulation selon qu'il y a eu impact ou non

In [22]:
# 1. sous-ensemble des locations chaînées à une précédente location du même véhicule (<12h)
chained = dfda[dfda['previous_ended_rental_id'].notna()].copy()
print(chained.shape)
chained.head()

(1841, 8)


,rental_id,car_id,checkin_type,state,delay_at_checkout_in_minutes,previous_ended_rental_id,time_delta_with_previous_rental_in_minutes,is_late
6,511639,370585,connect,ended,-15.0,563782.0,570.0,False
19,519491,312389,mobile,ended,58.0,545639.0,420.0,True
23,521156,392479,mobile,ended,NaN,537298.0,0.0,<NA>
34,525044,349751,mobile,ended,NaN,510607.0,60.0,<NA>
40,528808,181625,connect,ended,-76.0,557404.0,330.0,False


In [23]:
# 2. auto-jointure : récupérer le delay réel de la location précédente désignée par previous_ended_rental_id
# serie.map(other_serie) : chaque valeur de s est cherchée dans l'index de other_series, et remplacée par la valeur correspondante
# creation d'une serie indexé sur rental_id
delay_by_rental = dfda.set_index('rental_id')['delay_at_checkout_in_minutes']
# en fournissant une serie à map (au lieu d'une fonction) on cherche dans l'index de delay_by_rental, 
# et on remplace par la valeur correspondante (delay_at_checkout_in_minutes)
chained['previous_delay'] = chained['previous_ended_rental_id'].astype(int).map(delay_by_rental)

chained[['rental_id', 'previous_ended_rental_id', 'previous_delay', 'time_delta_with_previous_rental_in_minutes']].head()

,rental_id,previous_ended_rental_id,previous_delay,time_delta_with_previous_rental_in_minutes
6,511639,563782.0,136.0,570.0
19,519491,545639.0,140.0,420.0
23,521156,537298.0,NaN,0.0
34,525044,510607.0,-113.0,60.0
40,528808,557404.0,-352.0,330.0


In [24]:
# 3. le conducteur suivant est concrètement impacté si le delay réel de la précédente location
# dépasse l'écart prévu entre les deux réservations. NA si on ne connaît pas le delay précédent.
chained['checkin_impacted'] = (chained['previous_delay'] > chained['time_delta_with_previous_rental_in_minutes']).astype('boolean')
chained.loc[chained['previous_delay'].isna(), 'checkin_impacted'] = pd.NA

chained['checkin_impacted'].value_counts(dropna=False)

checkin_impacted
False    1511
True      218
<NA>      112
Name: count, dtype: Int64

In [25]:
checkin_impacted_counts = chained['checkin_impacted'].value_counts(dropna=False).rename_axis('checkin_impacted').reset_index(name='count')
checkin_impacted_counts['checkin_impacted'] = checkin_impacted_counts['checkin_impacted'].astype('object').map({True: 'impacté', False: 'non impacté'}).fillna('inconnu')

impacted_colors = {'impacté': '#EF553B', 'non impacté': '#00CC96', 'inconnu': '#BAB0AC'}
fig = px.pie(
    checkin_impacted_counts, names='checkin_impacted', values='count', color='checkin_impacted',
    color_discrete_map=impacted_colors, title='Répartition des locations chaînées par impact au checkin', width=400
)
fig.update_layout(legend=dict(orientation='h', yanchor='top', y=0.05, xanchor='center', x=0.5))
fig.show()

In [26]:
# 4. cas "problématiques" : taux d'annulation selon que le checkin suivant a été impacté ou non
pd.crosstab(chained['checkin_impacted'], chained['state'], normalize='index')

state,canceled,ended
checkin_impacted,,
False,0.111846,0.888154
True,0.169725,0.830275


In [27]:
cancel_rate_by_impact = (
    chained.dropna(subset=['checkin_impacted'])
    .groupby('checkin_impacted')['state']
    .apply(lambda s: (s == 'canceled').mean())
    .reset_index(name='taux_annulation')
)
cancel_rate_by_impact['checkin_impacted'] = cancel_rate_by_impact['checkin_impacted'].map({True: 'impacté', False: 'non impacté'})
cancel_rate_by_impact['pct_label'] = cancel_rate_by_impact['taux_annulation'].apply(lambda x: f"{x:.1%}")

fig = px.bar(
    cancel_rate_by_impact, x='checkin_impacted', y='taux_annulation', color='checkin_impacted',
    color_discrete_map=impacted_colors, text='pct_label',
    title="Taux d'annulation selon l'impact au checkin", width=450
)
fig.update_traces(textposition='outside')
fig.update_yaxes(title='taux d\'annulation')
fig.show()

In [28]:
# les voitures connect sont-elles sur/sous-représentées parmi les locations chaînées, par rapport à leur poids réel ?
print("répartition checkin_type - population globale :")
print(dfda['checkin_type'].value_counts(normalize=True))
print()
print("répartition checkin_type - locations chaînées :")
print(chained['checkin_type'].value_counts(normalize=True))

répartition checkin_type - population globale :
checkin_type
mobile     0.797888
connect    0.202112
Name: proportion, dtype: float64

répartition checkin_type - locations chaînées :
checkin_type
mobile     0.558392
connect    0.441608
Name: proportion, dtype: float64


In [29]:
chained_checkin_counts = chained['checkin_type'].value_counts().rename_axis('checkin_type').reset_index(name='count')

fig2 = px.pie(
    chained_checkin_counts, names='checkin_type', values='count', color='checkin_type',
    color_discrete_map=checkin_colors, title='checkin_type - locations chaînées', width=400
)
fig2.update_layout(legend=dict(orientation='h', yanchor='top', y=0.05, xanchor='center', x=0.5))
fig2.show()

---
Connect est **sur-représenté** parmi les locations chaînées (44.2% contre 20.2% dans la population globale, soit ~2.2x son poids réel) ; mobile est sous-représenté symétriquement (55.8% contre 79.8%). Les voitures connect s'enchaînent donc structurellement avec moins de marge entre deux réservations que les voitures mobile — ce qui explique pourquoi, à seuil égal, le scope `connect_only` affiche systématiquement un `pct_affected`/`pct_revenue_at_risk` proportionnellement plus élevé que le scope `all` dans la simulation plus bas (ce n'est pas un artefact de calcul, mais un vrai effet de composition).

---
218 cas (11.8%) réellement impactés 
taux d'annulation 16.97% vs 11.18% (+52% relatif) si impacté

### Note : `checkin_type` est-il fixe par voiture ?

Vérification nécessaire pour la simulation du scope `connect_only` : on veut s'assurer que filtrer sur `checkin_type` revient bien à filtrer sur un sous-parc de véhicules cohérent, et non sur un choix qui varierait location par location pour une même voiture.

In [44]:
# checkin_type est-il un attribut fixe de la voiture, ou peut-il varier d'une location à l'autre pour un même car_id ?
nb_checkin_types_per_car = dfda.groupby('car_id')['checkin_type'].nunique()
(nb_checkin_types_per_car > 1).sum(), nb_checkin_types_per_car.shape[0]

(96, 8143)

---
96 sur 8143 voitures (1.2%) ont plusieurs types → quasi-fixe, scope basé sur le checkin_type de la location suivante

## Prix journalier de référence (proxy, durée de location inconnue)

Les deux datasets n'ont aucune clé de jointure commune (le `car_id` du dataset delay ne correspond à aucun identifiant du dataset pricing), et aucun des deux ne renseigne la durée réelle des locations. On charge donc `get_around_pricing_project.csv` uniquement pour calculer un prix/jour médian, utilisé comme proxy uniforme dans la simulation qui suit en gardant à l'esprit que ça suppose implicitement 1 jour de location par réservation affectée.

In [30]:
# pas de clé de jointure entre les deux datasets : on utilise la médiane du prix/jour comme proxy de revenu
dfp = pd.read_csv("data/get_around_pricing_project.csv", index_col=0)
print(dfp.shape)
display(dfp.head())

median_price_per_day = dfp['rental_price_per_day'].median()
print(f"prix median : {median_price_per_day}")

(4843, 14)


,model_key,mileage,engine_power,fuel,paint_color,car_type,private_parking_available,has_gps,has_air_conditioning,automatic_car,has_getaround_connect,has_speed_regulator,winter_tires,rental_price_per_day
0,Citroën,140411,100,diesel,black,convertible,True,True,False,False,True,True,True,106
1,Citroën,13929,317,petrol,grey,convertible,True,True,False,False,False,True,True,264
2,Citroën,183297,120,diesel,white,convertible,False,False,False,False,True,False,True,101
3,Citroën,128035,135,diesel,red,convertible,True,True,False,False,True,True,True,158
4,Citroën,97097,160,diesel,silver,convertible,True,True,False,False,False,True,True,183


prix median : 119.0


## Simulation seuil / scope

Pour chaque combinaison (seuil de délai minimum, scope `all`/`connect_only`), on calcule sur le sous-ensemble `chained` : la part de locations qui auraient été bloquées, le revenu estimé à risque (via le prix proxy), et la part des 218 cas problématiques (`checkin_impacted`) qui auraient été évités.

In [31]:
def simulate(threshold, scope):
    """
    Simule l'impact d'un seuil de tolérance sur les locations chaînées.
    threshold : seuil de tolérance en minutes
    scope : 'connect_only' pour ne considérer que les locations connect, 'all' pour toutes les locations
    """
    # deux masques de scope : un sur dfda (population totale, pour les dénominateurs des %),
    # un sur chained (le seul sous-ensemble concerné par un blocage, car seules ces locations ont un time_delta connu)
    if scope == 'connect' or scope == 'mobile':
        scope_mask_full = dfda['checkin_type'] == scope
        scope_mask_chained = chained['checkin_type'] == scope
    else:
        scope_mask_full = pd.Series(True, index=dfda.index)
        scope_mask_chained = pd.Series(True, index=chained.index)

    # volume total de locations dans le scope (dénominateur de pct_affected)
    total_rentals = scope_mask_full.sum()
    # revenu total proxy du scope : seules les locations "ended" génèrent du revenu (les canceled non)
    # même hypothèse "1 jour = 1 location" que revenue_at_risk plus bas, donc le ratio des deux reste comparable
    total_ended_revenue = (scope_mask_full & (dfda['state'] == 'ended')).sum() * median_price_per_day

    # locations qui n'auraient pas pu être réservées si le seuil imposé était `threshold`
    # (écart réellement prévu avec la location précédente < seuil demandé)
    affected_mask = scope_mask_chained & (chained['time_delta_with_previous_rental_in_minutes'] < threshold)
    n_affected = affected_mask.sum()
    pct_affected = n_affected / total_rentals * 100

    # revenu à risque = nb de locations bloquées × prix/jour proxy (voir limite de durée inconnue, section précédente)
    revenue_at_risk = n_affected * median_price_per_day
    pct_revenue_at_risk = revenue_at_risk / total_ended_revenue * 100

    # cas historiquement problématiques : le checkin suivant a réellement été retardé (checkin_impacted == True)
    # ce nombre est fixe pour un scope donné, il ne dépend pas du seuil testé
    problematic_mask = scope_mask_chained & (chained['checkin_impacted'] == True)
    n_problematic = problematic_mask.sum()
    # parmi ces cas problématiques, ceux que le seuil aurait empêchés en amont (réservation jamais autorisée)
    n_resolved = (problematic_mask & (chained['time_delta_with_previous_rental_in_minutes'] < threshold)).sum()
    n_remaining = n_problematic - n_resolved
    pct_problematic_resolved = n_resolved / n_problematic * 100

    return {
        'threshold': threshold,
        'scope': scope,
        'total_rentals': total_rentals,
        'n_affected': n_affected,
        'pct_affected': pct_affected,
        'revenue_at_risk': revenue_at_risk,
        'pct_revenue_at_risk': pct_revenue_at_risk,
        'n_problematic': n_problematic,
        'n_resolved': n_resolved,
        'n_remaining': n_remaining,
        'pct_problematic_resolved': pct_problematic_resolved,
    }

In [ ]:
# grille alignée sur le pas réel des données (paliers de 30 min), jusqu'au max observé
max_time_delta = int(chained['time_delta_with_previous_rental_in_minutes'].max())
print(f"max time delta : {max_time_delta} min")
thresholds = list(range(0, max_time_delta + 30, 30))
scopes = ['all', 'connect', 'mobile']

results = pd.DataFrame([simulate(t, s) for t in thresholds for s in scopes])
display(results)

max time delta : 720 min


,threshold,scope,total_rentals,n_affected,pct_affected,revenue_at_risk,pct_revenue_at_risk,n_problematic,n_resolved,n_remaining,pct_problematic_resolved
0,0,all,21310,0,0.000000,0.0,0.000000,218,0,218,0.000000
1,0,connect,4307,0,0.000000,0.0,0.000000,69,0,69,0.000000
2,0,mobile,17003,0,0.000000,0.0,0.000000,149,0,149,0.000000
3,30,all,21310,279,1.309244,33201.0,1.546135,218,116,102,53.211009
4,30,connect,4307,131,3.041560,15589.0,3.733257,69,40,29,57.971014
...,...,...,...,...,...,...,...,...,...,...,...
70,690,connect,4307,723,16.786626,86037.0,20.604161,69,69,0,100.000000
71,690,mobile,17003,923,5.428454,109837.0,6.349752,149,147,2,98.657718
72,720,all,21310,1711,8.029094,203609.0,9.481851,218,216,2,99.082569
73,720,connect,4307,755,17.529603,89845.0,21.516101,69,69,0,100.000000


### Visualisation des résultats

Évolution du % de locations affectées et du nombre de cas problématiques résolus en fonction du seuil, pour les deux scopes.

In [53]:
fig = px.line(
    results, x='threshold', y='pct_affected', color='scope', markers=True,
    title='% de locations affectées selon le seuil et le type de check-in',
    width=600,
    height=600
)
fig.update_xaxes(dtick=60, title='Seuil (min)')
fig.update_yaxes(title='% de locations affectées')
fig.update_layout(legend_title_text='Type de check-in')
fig.show()

In [57]:
fig = px.line(
    results, x='threshold', y='pct_problematic_resolved', color='scope', markers=True,
    title='% de locations problématiques résolus selon le seuil et le type de check-in',
    width=600,
    height=600
)
fig.update_xaxes(dtick=60, title='Seuil (min)')
fig.update_yaxes(title='% de cas problématiques résolus')
fig.update_layout(legend_title_text='Type de check-in')
fig.show()

## Conclusion : réponses aux questions du PM

**1. Part du revenu potentiellement affecté ?**
Estimée via un prix/jour proxy (médiane 119€, faute de clé de jointure entre les deux datasets). À un seuil de 120 min (scope `all`), ~3.7% du revenu estimé serait affecté ; en scope `connect`, ce taux monte à ~8.4% (les locations Connect s'enchaînent avec moins de marge, donc proportionnellement plus touchées — voir point 3).

*Limite à garder en tête* : ni le dataset delay ni le dataset pricing ne renseignent la durée réelle des locations. Les montants en € affichés dans `results` sont donc des équivalents "1 jour de location par réservation affectée", pas un revenu réel — à ne pas citer comme un montant exact. Le `%` (`pct_revenue_at_risk`) reste la lecture la plus fiable des deux, car il applique la même hypothèse de durée au numérateur et au dénominateur.

**2. Combien de locations affectées selon seuil et scope ?**
Voir le tableau `results` : de 0% (seuil=0) à ~8% des locations sur l'ensemble de la flotte au seuil max observé (720 min). La progression est nette jusqu'à ~180-240 min, puis les gains marginaux ralentissent fortement.

**3. Fréquence des retards impactant le prochain conducteur ?**
Sur les 1 841 locations chaînées à une précédente location du même véhicule (<12h d'écart), 218 (11.8%) ont un checkin réellement impacté par le retard de la location précédente. Ces cas impactés ont un taux d'annulation nettement plus élevé (16.97% vs 11.18%), soit +52% relatif.

En comparant les scopes `connect` et `mobile` séparément (plutôt que `connect` vs `all`), le rapport coût/bénéfice de Connect apparaît clairement défavorable : à seuil comparable (≤240 min), Connect coûte environ **3x plus cher** que mobile en part de son propre parc affecté (ex. à 120 min : 6.85% vs 2.18% ; à 240 min : 9.98% vs 3.36%), pour à peine **4 points de résolution en plus** des cas problématiques (85.5% vs 81.2% à 120 min ; 95.7% vs 91.3% à 240 min). Au-delà de 270 min, le classement se brouille (Connect stagne à 97.1% entre 270 et 540 min, mobile continue de progresser doucement et le dépasse temporairement), mais cette zone est de toute façon hors du seuil pertinent identifié au point 4.

**4. Combien de cas problématiques résolus selon le seuil choisi ?**
À 120 min (scope `all`) : 180/218 cas résolus (82.6%). Au-delà de 240 min, on ne gagne plus que quelques points pour un coût croissant en revenu affecté.

**Recommandation** : seuil autour de **120 minutes**, appliqué à **tous les véhicules** (pas seulement Connect) — le scope `connect` seul laisserait 149 des 218 cas problématiques (68.3%, ceux survenant en mobile) sans solution, quel que soit le seuil. C'est d'autant plus vrai que Connect, en plus d'être sur-représenté dans les locations à risque de chaînage et ~3x plus coûteux à traiter que mobile pour un gain de résolution comparable, ne concentre que 69 des 218 cas réellement problématiques (31.7%) — limiter la feature à Connect pénaliserait donc proportionnellement plus ce segment sans pour autant traiter la majorité des cas problématiques.